# LightGBM at 75,000 training rows

Fits the default and resumable Optuna-tuned LightGBM baselines. A CPU runtime is sufficient.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_REPO_URL='https://github.com/selimsidan/tfm-credit-benchmark.git'
GITHUB_BRANCH='main'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/tfm_credit_benchmark'
SOURCE_EXPERIMENT_NAME='lending_club_900k'
TRIAL_EXPERIMENT_NAME='lending_club_75k_trial'
FORCE_RERUN=False
RUN_TESTS=True
LIGHTGBM_TUNING_MINUTES=15
MAX_OPTUNA_TRIALS=30
# ==================================================================


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys
drive.mount('/content/drive'); repo=Path('/content/tfm-credit-benchmark')
if (repo/'.git').is_dir(): subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH],check=True)
else: subprocess.run(['git','clone','--branch',GITHUB_BRANCH,'--single-branch',GITHUB_REPO_URL,str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
src_path=str(repo/'src')
if src_path not in sys.path: sys.path.insert(0,src_path)
import tfmbench
print('tfmbench package:',tfmbench.__file__)
print('Repository commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())


In [ ]:
import json
from tfmbench.resources import hardware_info
from tfmbench.trial import TrialConfig, verify_trial_manifest
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
print(json.dumps(hardware_info(),indent=2))
config=TrialConfig(DRIVE_OUTPUT_DIR,TRIAL_EXPERIMENT_NAME,SOURCE_EXPERIMENT_NAME)
manifest=verify_trial_manifest(config); print('Verified trial manifest:',manifest['manifest_hash'])


In [ ]:
from tfmbench.trial import run_trial_method
table=run_trial_method('lgbm',DRIVE_OUTPUT_DIR,SOURCE_EXPERIMENT_NAME,TRIAL_EXPERIMENT_NAME,force_rerun=FORCE_RERUN,tuning_minutes=LIGHTGBM_TUNING_MINUTES,max_trials=MAX_OPTUNA_TRIALS)


In [ ]:
display(table)
print('Default results:',config.experiment_root/'lgbm_default'/'n75000')
print('Tuned results:',config.experiment_root/'lgbm_tuned'/'n75000')
print('Next notebook: 08_realmlp_75k.ipynb')
